# InSAR deformation profile: 2023 Türkiye–Syria earthquake (Mw 7.8)

Co-seismic deformation from a Sentinel-1 interferogram (COMET-LiCSAR, 29 Jan to 10 Feb 2023) spanning the 6 February 2023 earthquakes. The notebook samples wrapped and unwrapped phase along a line across the deformation zone, measures distance along it on the WGS84 ellipsoid, and uses coherence to flag unreliable samples.

**Author:** Shaghayegh Zare. Developed from Lab 4 of *Satellite Geodesy Observation Techniques* (University of Stuttgart, WS 2025/26, team work with Haibo and Shiwei); code rewritten and restructured for this repository.

Input GeoTIFFs go in `data/` (see the README). Figures are written to `figures/`.

## Settings

In [ ]:
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import rasterio
from rasterio.transform import xy
from pyproj import Geod

DATA_DIR = Path("data")
FIG_DIR = Path("figures")
FIG_DIR.mkdir(exist_ok=True)

PAIR = "20230129_20230210"  # LiCSAR interferogram, primary_secondary date
FILES = {
    "wrapped": DATA_DIR / f"{PAIR}.geo.diff_pha.tif",
    "unwrapped": DATA_DIR / f"{PAIR}.geo.unw.tif",
    "coherence": DATA_DIR / f"{PAIR}.geo.cc.tif",
}

# Profile line in pixel coordinates (column, row): from the south-east end,
# across the main fringe pattern, roughly perpendicular to the rupture
PROFILE_START = (2500, 4300)
PROFILE_END = (1750, 600)
N_SAMPLES = 2000

MIN_COHERENCE = 0.2  # samples below this value are treated as unreliable

## Helper functions

In [ ]:
def read_band(path):
    """Return the first band of a GeoTIFF and its affine transform."""
    with rasterio.open(path) as src:
        return src.read(1), src.transform


def to_unit_coherence(raw):
    """LiCSAR stores coherence as 8-bit integers; rescale to 0-1."""
    return raw / 255.0 if raw.dtype == np.uint8 else raw.astype(float)


def line_pixels(start, end, n):
    """Integer (row, col) indices of n points on a straight line in the image."""
    cols = np.linspace(start[0], end[0], n).astype(int)
    rows = np.linspace(start[1], end[1], n).astype(int)
    return rows, cols


def along_track_km(transform, rows, cols):
    """Cumulative WGS84 geodesic distance (km) through the pixel centres."""
    lon, lat = (np.asarray(v) for v in xy(transform, rows, cols))
    _, _, step_m = Geod(ellps="WGS84").inv(lon[:-1], lat[:-1], lon[1:], lat[1:])
    return np.concatenate([[0.0], np.cumsum(step_m)]) / 1000


def masked_nodata(arr):
    """Hide zero-valued no-data pixels in map plots (display only)."""
    return np.ma.masked_equal(arr, 0)


def show_map(ax, arr, title, cmap, label, **kwargs):
    im = ax.imshow(masked_nodata(arr), cmap=cmap, interpolation="nearest", **kwargs)
    ax.set_title(title)
    ax.set_xlabel("Column (px)")
    ax.set_ylabel("Row (px)")
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label=label)
    return im


def mark_profile(ax):
    (c0, r0), (c1, r1) = PROFILE_START, PROFILE_END
    ax.plot([c0, c1], [r0, r1], color="black", linewidth=2)
    ax.annotate("start", (c0, r0), xytext=(6, -2), textcoords="offset points")


def phase_with_coherence(ax, dist, phase, coherence, color, title):
    """Phase on the left axis, coherence on a shared right axis."""
    ax.plot(dist, phase, ".", color=color, markersize=2)
    ax.set_xlabel("Distance along profile (km)")
    ax.set_ylabel("Phase (rad)", color=color)
    ax.tick_params(axis="y", labelcolor=color)
    ax.grid(alpha=0.4)
    ax.set_title(title)
    right = ax.twinx()
    right.plot(dist, coherence, color="tab:red", linewidth=0.8, alpha=0.5)
    right.axhline(MIN_COHERENCE, color="tab:red", linestyle=":", linewidth=1)
    right.set_ylim(0, 1)
    right.set_ylabel("Coherence", color="tab:red")
    right.tick_params(axis="y", labelcolor="tab:red")
    return right


def save(fig, name):
    fig.savefig(FIG_DIR / name, dpi=150, bbox_inches="tight")

## 1. Read the interferogram

In [ ]:
wrapped, transform = read_band(FILES["wrapped"])
coherence = to_unit_coherence(read_band(FILES["coherence"])[0])
unwrapped = read_band(FILES["unwrapped"])[0] if FILES["unwrapped"].exists() else None

print(f"Grid: {wrapped.shape[0]} rows x {wrapped.shape[1]} columns")
if unwrapped is None:
    print("No unwrapped phase file found; step 5 is skipped.")

## 2. Maps

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
show_map(ax, wrapped, "Wrapped phase", "twilight", "Phase (rad)", vmin=-np.pi, vmax=np.pi)
save(fig, "wrapped_phase.png")
plt.show()

fig, ax = plt.subplots(figsize=(8, 6))
show_map(ax, coherence, "Coherence", "viridis", "Coherence", vmin=0, vmax=1)
save(fig, "coherence_map.png")
plt.show()

## 3. Sample the profile

In [ ]:
rows, cols = line_pixels(PROFILE_START, PROFILE_END, N_SAMPLES)
dist_km = along_track_km(transform, rows, cols)

wrapped_p = wrapped[rows, cols]
coherence_p = coherence[rows, cols]

print(f"Profile length: {dist_km[-1]:.1f} km, {N_SAMPLES} samples, "
      f"mean spacing {1000 * dist_km[-1] / (N_SAMPLES - 1):.0f} m")

## 4. Coherence check

In [ ]:
reliable = coherence_p >= MIN_COHERENCE
n_ok = int(reliable.sum())
print(f"Coherence >= {MIN_COHERENCE}: {n_ok} of {reliable.size} samples kept "
      f"({100 * n_ok / reliable.size:.0f} %), {reliable.size - n_ok} masked")

fig, ax = plt.subplots(figsize=(15, 3))
phase_with_coherence(ax, dist_km[reliable], wrapped_p[reliable], coherence_p[reliable],
                     "tab:blue", f"Wrapped phase where coherence >= {MIN_COHERENCE}")
save(fig, "filtered_profile.png")
plt.show()

## 5. Wrapped vs unwrapped phase along the profile

In [ ]:
if unwrapped is None:
    print("Skipped: no unwrapped phase.")
else:
    unwrapped_p = unwrapped[rows, cols]
    has_unw = np.isfinite(unwrapped_p) & (unwrapped_p != 0)  # 0 = no data in LiCSAR

    fig, axes = plt.subplots(2, 2, figsize=(14, 8))
    show_map(axes[0, 0], wrapped, "Wrapped phase", "twilight", "Phase (rad)",
             vmin=-np.pi, vmax=np.pi)
    show_map(axes[0, 1], unwrapped, "Unwrapped phase", "RdBu_r", "Phase (rad)")
    for ax in axes[0]:
        mark_profile(ax)

    phase_with_coherence(axes[1, 0], dist_km, wrapped_p, coherence_p,
                         "tab:blue", "Wrapped phase")
    phase_with_coherence(axes[1, 1], dist_km[has_unw], unwrapped_p[has_unw],
                         coherence_p[has_unw], "tab:green", "Unwrapped phase")

    fig.suptitle("Phase along the profile (dotted red line: coherence threshold)", fontsize=13)
    fig.tight_layout(rect=[0, 0, 1, 0.96])
    save(fig, "wrapped_vs_unwrapped_profile.png")
    plt.show()